# Le cube OLAP (DuckDB)

Ici on construit le **cube OLAP** des défaillances. Il sert à deux choses (voir `docs/axes_analyse.md`) :
- **explorer** rapidement les données (coupes, passage d'un niveau à l'autre) ;
- **pré-agréger** les comptages une fois pour toutes. Le dashboard Dash et les analyses (graphes, clustering) lisent ces petits tableaux au lieu de relire les 3,3 M de lignes du BODACC.

On utilise **DuckDB** : c'est une base de données qui travaille directement sur les fichiers parquet, avec du SQL. Elle sait calculer tous les sous-totaux d'un cube d'un seul coup avec `GROUP BY CUBE (...)`.

⚠️ **RAM** : on limite DuckDB à 4 Go. S'il a besoin de plus, il passe par le disque au lieu de faire planter la machine.

### Lexique

| Terme | En clair |
|---|---|
| **Table de faits** | La table de base du cube : une ligne par événement BODACC et par entreprise, avec toutes ses caractéristiques. |
| **Dimension** | Un axe pour découper les comptages : le temps, le territoire, le secteur… |
| **Hiérarchie** | Les niveaux d'une dimension, du plus large au plus fin : région → département → tribunal. |
| **Mesure** | Ce qu'on compte : le nombre d'événements, le nombre d'entreprises. |
| **`CUBE`** | Calcule les sous-totaux pour **toutes les combinaisons** de dimensions (avec 3 dimensions : 2³ = 8 combinaisons). |
| **`ROLLUP`** | Calcule les sous-totaux en suivant une **hiérarchie** (mois → trimestre → année → total). |
| **« Total »** | Dans les tableaux du cube, « Total » veut dire « toutes les valeurs de cette dimension réunies ». |

### Les décisions (Luka)

| Sujet | Décision |
|---|---|
| Outil | **DuckDB** |
| Territoire | **Département du BODACC** (celui de l'annonce), rempli pour toutes les annonces, y compris celles sans SIREN. La région est déduite du département. |
| Âge | Âge de l'entreprise à la **date du jugement**, en 6 tranches : moins d'1 an, 1-3 ans, 3-5 ans, 5-10 ans, 10-20 ans, 20 ans et plus. |
| Taille | **Chiffre d'affaires winsorisé** des derniers comptes clôturés avant le jugement (36 mois maximum), en 5 tranches : moins de 100 k€, 100-500 k€, 500 k€-2 M€, 2-10 M€, 10 M€ et plus. |
| Taux de défaillance | Rapporté aux entreprises **actives au 1er janvier** de l'année : créées avant le 1er janvier et pas encore cessées à cette date (entreprises purgées comprises). |

## 1. Préparation

In [1]:
import sys
sys.path.append("../scripts")

import duckdb
import pandas as pd
from pathlib import Path
from data_utils import PROCESSED, RAW, mem

DASH = PROCESSED.parent / "dashboard"
DASH.mkdir(exist_ok=True)
pd.set_option("display.max_rows", 60)

con = duckdb.connect()
con.execute("SET memory_limit = '4GB'")
con.execute("SET threads = 4")
con.execute("SET preserve_insertion_order = false")
con.execute(f"SET temp_directory = '{PROCESSED.parent / 'duckdb_tmp'}'")

def sql(q):
    """Lance une requête et renvoie le résultat en DataFrame."""
    return con.execute(q).df()

print("DuckDB", duckdb.__version__, "|", mem())

DuckDB 2.0.0 | process : 0.18 Go | disponible : 10.52 Go


## 2. Les régions

On fabrique la correspondance **département → région** à partir des annonces BODACC récentes (2017 et après, donc avec les régions actuelles). On s'en sert pour le BODACC **et** pour SIRENE, comme ça les deux parlent des mêmes régions.

In [2]:
REGIONS = {1: "Guadeloupe", 2: "Martinique", 3: "Guyane", 4: "La Réunion", 6: "Mayotte",
           11: "Île-de-France", 24: "Centre-Val de Loire", 27: "Bourgogne-Franche-Comté", 28: "Normandie",
           32: "Hauts-de-France", 44: "Grand Est", 52: "Pays de la Loire", 53: "Bretagne",
           75: "Nouvelle-Aquitaine", 76: "Occitanie", 84: "Auvergne-Rhône-Alpes",
           93: "Provence-Alpes-Côte d'Azur", 94: "Corse", 975: "Saint-Pierre-et-Miquelon"}

fichiers = sorted((RAW / "bodacc").glob("procedures_collectives_20*.csv.gz"))
recents = [f for f in fichiers if int(f.stem[-8:-4]) >= 2017]
dep = pd.concat([pd.read_csv(f, sep=";", dtype=str, usecols=["numerodepartement", "region_code"]) for f in recents])
dep_region = (dep.groupby("numerodepartement")["region_code"].agg(lambda s: s.mode().iloc[0])
                 .astype(int).map(REGIONS).rename("region").rename_axis("departement").reset_index())
dep_region.loc[dep_region["departement"] == "985", "region"] = "Mayotte"  # 985 : ancien code de Mayotte (tribunal de Mamoudzou)
del dep
con.register("dep_region", dep_region)
print(len(dep_region), "départements | sans région :", dep_region["region"].isna().sum())
dep_region.groupby("region").size().sort_values(ascending=False)

103 départements | sans région : 1


region
Occitanie                     13
Nouvelle-Aquitaine            12
Auvergne-Rhône-Alpes          12
Grand Est                     10
Bourgogne-Franche-Comté        8
Île-de-France                  8
Provence-Alpes-Côte d'Azur     6
Centre-Val de Loire            6
Normandie                      5
Hauts-de-France                5
Pays de la Loire               5
Bretagne                       4
Corse                          2
Guyane                         1
Guadeloupe                     1
Martinique                     1
La Réunion                     1
Mayotte                        1
Saint-Pierre-et-Miquelon       1
dtype: int64

### Observations

- On retrouve les **18 régions** actuelles plus Saint-Pierre-et-Miquelon, pour 103 départements.
- Le code **985** est l'ancien code de **Mayotte** (ses annonces viennent du tribunal de Mamoudzou) : on le rattache à Mayotte. Seul le 986 (Wallis-et-Futuna) n'a pas de région : il ne concerne que **2 annonces**, notées « non renseigné ».

## 3. La table de faits

Une ligne par événement BODACC et par entreprise (`bodacc_clean.parquet`), à laquelle on ajoute :
- les infos **SIRENE** de l'entreprise : secteur, forme juridique, date de création ;
- la **taille** : le chiffre d'affaires des derniers comptes clôturés **avant** l'événement, s'ils ont 36 mois maximum. Pour trouver « les derniers comptes avant une date », on utilise un `ASOF JOIN` : pour chaque événement, DuckDB prend l'exercice le plus récent qui le précède.

Les valeurs inconnues sont notées « non renseigné », pour ne pas les confondre avec les « Total » du cube.

In [3]:
con.execute(f"""
CREATE OR REPLACE TABLE faits AS
WITH ev AS (
    SELECT b.*, CAST(b.date_evenement AS TIMESTAMP) AS date_ev
    FROM read_parquet('{PROCESSED / "bodacc_clean.parquet"}') b
),
ratios AS (
    SELECT siren, CAST(date_cloture_exercice AS TIMESTAMP) AS cloture, chiffre_d_affaires AS ca
    FROM read_parquet('{PROCESSED / "ratios_clean.parquet"}')
),
sirene AS (
    SELECT siren, date_creation, CAST(cj_famille AS VARCHAR) AS cj_famille,
           CAST(categorie_juridique AS VARCHAR) AS categorie_juridique,
           CAST(naf_section AS VARCHAR) AS naf_section, CAST(naf_division AS VARCHAR) AS naf_division
    FROM read_parquet('{PROCESSED / "sirene_clean.parquet"}')
),
joint AS (
    SELECT ev.*, s.date_creation, s.cj_famille, s.categorie_juridique, s.naf_section, s.naf_division,
           CASE WHEN r.cloture >= ev.date_ev - INTERVAL 36 MONTH THEN r.ca END AS ca
    FROM ev
    LEFT JOIN sirene s USING (siren)
    ASOF LEFT JOIN ratios r ON ev.siren = r.siren AND ev.date_ev > r.cloture
),
age AS (
    SELECT *, CASE WHEN date_ev >= date_creation THEN datediff('day', date_creation, date_ev) / 365.25 END AS age_ans
    FROM joint
)
SELECT
    siren,
    date_ev AS date_evenement,
    year(date_ev) AS annee,
    year(date_ev) || '-T' || quarter(date_ev) AS trimestre,
    strftime(date_ev, '%Y-%m') AS mois,
    coalesce(d.region, 'non renseigné') AS region,
    CAST(numerodepartement AS VARCHAR) AS departement,
    CAST(tribunal AS VARCHAR) AS tribunal,
    coalesce(naf_section, 'non renseigné') AS naf_section,
    coalesce(naf_division, 'non renseigné') AS naf_division,
    coalesce(cj_famille, 'non renseigné') AS cj_famille,
    coalesce(categorie_juridique, 'non renseigné') AS categorie_juridique,
    CASE WHEN age_ans IS NULL THEN 'non renseigné'
         WHEN age_ans < 1 THEN '< 1 an' WHEN age_ans < 3 THEN '1-3 ans' WHEN age_ans < 5 THEN '3-5 ans'
         WHEN age_ans < 10 THEN '5-10 ans' WHEN age_ans < 20 THEN '10-20 ans' ELSE '20 ans et +' END AS tranche_age,
    CASE WHEN ca IS NULL THEN 'non renseigné'
         WHEN ca < 100000 THEN '< 100 k€' WHEN ca < 500000 THEN '100-500 k€' WHEN ca < 2000000 THEN '500 k€-2 M€'
         WHEN ca < 10000000 THEN '2-10 M€' ELSE '10 M€ et +' END AS tranche_taille,
    CASE WHEN siren IS NULL THEN 'sans SIREN'
         WHEN CAST(type_personne AS VARCHAR) = 'pm' THEN 'personne morale'
         WHEN CAST(type_personne AS VARCHAR) = 'pp' THEN 'personne physique'
         ELSE 'non renseigné' END AS type_personne,
    CAST(categorie AS VARCHAR) AS categorie,
    CAST(nature AS VARCHAR) AS nature,
    age_ans,
    ca AS chiffre_d_affaires,
    annonce_collective
FROM age
LEFT JOIN dep_region d ON CAST(age.numerodepartement AS VARCHAR) = d.departement
""")
print(sql("SELECT count(*) AS lignes FROM faits"), "|", mem())
sql("SELECT * FROM faits LIMIT 5")

    lignes
0  3340053 | process : 1.05 Go | disponible : 9.46 Go


,siren,date_evenement,annee,trimestre,mois,region,departement,tribunal,naf_section,naf_division,cj_famille,categorie_juridique,tranche_age,tranche_taille,type_personne,categorie,nature,age_ans,chiffre_d_affaires,annonce_collective
0,<NA>,2007-03-20,2007,2007-T1,2007-03,Grand Est,67,TRIBUNAL JUDICIAIRE - SAVERNE,non renseigné,non renseigné,non renseigné,non renseigné,non renseigné,non renseigné,sans SIREN,issue,Jugement de clôture pour insuffisance d'actif,NaN,NaN,False
1,<NA>,2007-03-20,2007,2007-T1,2007-03,Grand Est,67,TRIBUNAL JUDICIAIRE - SAVERNE,non renseigné,non renseigné,non renseigné,non renseigné,non renseigné,non renseigné,sans SIREN,issue,Jugement de clôture pour insuffisance d'actif,NaN,NaN,False
2,<NA>,2007-04-02,2007,2007-T2,2007-04,Grand Est,67,TRIBUNAL JUDICIAIRE - STRASBOURG,non renseigné,non renseigné,non renseigné,non renseigné,non renseigné,non renseigné,sans SIREN,autre,Dépôt de l'état des créances,NaN,NaN,False
3,<NA>,2007-04-03,2007,2007-T2,2007-04,Hauts-de-France,59,TRIBUNAL JUDICIAIRE - HAZEBROUCK,non renseigné,non renseigné,non renseigné,non renseigné,non renseigné,non renseigné,sans SIREN,autre,Dépôt de l'état des créances,NaN,NaN,False
4,<NA>,2007-04-03,2007,2007-T2,2007-04,Hauts-de-France,59,TRIBUNAL JUDICIAIRE - HAZEBROUCK,non renseigné,non renseigné,non renseigné,non renseigné,non renseigné,non renseigné,sans SIREN,autre,Dépôt du projet de répartition,NaN,NaN,False


In [4]:
# Part de 'non renseigné' par dimension
dims = ["region", "naf_section", "cj_famille", "tranche_age", "tranche_taille", "type_personne"]
sql(" UNION ALL ".join(
    f"SELECT '{d}' AS dimension, avg(CASE WHEN {d} = 'non renseigné' THEN 1 ELSE 0 END) AS part_non_renseigne FROM faits"
    for d in dims))

,dimension,part_non_renseigne
0,naf_section,0.144093
1,region,0.000005
2,cj_famille,0.095407
3,tranche_age,0.095627
4,tranche_taille,0.957834
5,type_personne,0.000000


In [5]:
# La taille, pour les entrées en défaillance avec SIREN
sql("""
SELECT tranche_taille, count(*) AS entrees
FROM faits WHERE categorie = 'entrée en défaillance' AND siren IS NOT NULL
GROUP BY ALL ORDER BY entrees DESC
""")

,tranche_taille,entrees
0,non renseigné,874317
1,100-500 k€,20307
2,< 100 k€,16855
3,500 k€-2 M€,12526
4,2-10 M€,4860
5,10 M€ et +,1353


### Observations

- La table de faits a bien **3 340 053 lignes**, comme `bodacc_clean`.
- La forme juridique manque pour 10 % des lignes : ce sont les annonces **sans SIREN** (9 %), qu'on ne peut pas relier à SIRENE. Le secteur manque un peu plus souvent (14 %), parce qu'il faut ajouter les entreprises sans code NAFRev2.
- ⚠️ La **taille** n'est connue que pour **4 %** des lignes. Pour les entrées en défaillance avec SIREN, on a 55 901 tailles connues sur 930 218. C'est logique : les ratios ne commencent qu'en 2016, donc **aucune taille avant 2016**. Ensuite, la part connue va de 6 % (2017) à 17 % (2019), puis redescend à environ 10 % (2025-2026).

In [6]:
con.execute(f"COPY faits TO '{PROCESSED / 'cube_faits.parquet'}' (FORMAT parquet)")
print("table de faits enregistrée :", (PROCESSED / "cube_faits.parquet").stat().st_size / 1e6, "Mo")

table de faits enregistrée : 50.273366 Mo


## 4. Le cube principal

On croise **7 dimensions** au niveau le plus large : année, région, section NAF, famille juridique, tranche d'âge, tranche de taille, type de personne. Le **type d'événement** n'est pas « cubé » : il est toujours séparé, parce qu'additionner des entrées en défaillance et des clôtures n'a pas de sens.

Mesures :
- `nb_evenements` : le nombre d'événements ;
- `nb_entreprises` : le nombre d'**entreprises distinctes** (SIREN). Cette mesure ne s'additionne pas (une entreprise peut avoir plusieurs événements), c'est justement pour ça qu'on la pré-calcule pour chaque combinaison.

Calculer d'un coup les 2⁷ = 128 combinaisons avec `GROUP BY CUBE` dépasse les 4 Go, à cause du comptage des SIREN distincts. On calcule donc les 128 combinaisons **une par une** et on les empile : le résultat est exactement le même qu'un `CUBE`, mais chaque requête reste petite.

In [7]:
DIMS = ["annee", "region", "naf_section", "cj_famille", "tranche_age", "tranche_taille", "type_personne"]

def select_dims(dims):
    """Remplace le NULL des sous-totaux par 'Total' (GROUPING = 1 veut dire : dimension agrégée)."""
    return ", ".join(f"CASE WHEN GROUPING({d}) = 1 THEN 'Total' ELSE CAST({d} AS VARCHAR) END AS {d}" for d in dims)

from itertools import combinations

def cube_par_morceaux(nom, dims, fixes="categorie"):
    """Équivalent de GROUP BY fixes, CUBE(dims), calculé combinaison par combinaison."""
    con.execute(f"DROP TABLE IF EXISTS {nom}")
    for k in range(len(dims) + 1):
        for groupe in combinations(dims, k):
            cols = ", ".join(f"CAST({d} AS VARCHAR) AS {d}" if d in groupe else f"'Total' AS {d}" for d in dims)
            by = ", ".join([fixes, *groupe])
            q = f"SELECT {fixes}, {cols}, count(*) AS nb_evenements, count(DISTINCT siren) AS nb_entreprises FROM faits GROUP BY {by}"
            if con.execute(f"SELECT count(*) FROM information_schema.tables WHERE table_name = '{nom}'").fetchone()[0]:
                con.execute(f"INSERT INTO {nom} {q}")
            else:
                con.execute(f"CREATE TABLE {nom} AS {q}")

cube_par_morceaux("cube_principal", DIMS)
print(sql("SELECT count(*) AS lignes FROM cube_principal"), "|", mem())

    lignes
0  2987190 | process : 2.20 Go | disponible : 9.01 Go


In [8]:
# Exemple : les entrées en défaillance par année, tout le reste réuni
sql("""
SELECT annee, nb_evenements, nb_entreprises FROM cube_principal
WHERE categorie = 'entrée en défaillance' AND annee <> 'Total'
  AND region = 'Total' AND naf_section = 'Total' AND cj_famille = 'Total'
  AND tranche_age = 'Total' AND tranche_taille = 'Total' AND type_personne = 'Total'
ORDER BY annee
""")

,annee,nb_evenements,nb_entreprises
0,2007,1810,1415
1,2008,55684,48041
2,2009,62320,54857
3,2010,58961,52364
4,2011,58213,51925
5,2012,59113,53118
6,2013,60592,54746
7,2014,58825,53084
8,2015,59441,53461
9,2016,56253,50246


In [9]:
# Vérification : le total d'une coupe doit être la somme de ses morceaux (pour les événements)
sql("""
WITH t AS (SELECT * FROM cube_principal WHERE categorie = 'entrée en défaillance'
           AND region = 'Total' AND naf_section = 'Total' AND cj_famille = 'Total'
           AND tranche_age = 'Total' AND tranche_taille = 'Total' AND type_personne = 'Total')
SELECT (SELECT nb_evenements FROM t WHERE annee = 'Total') AS total,
       (SELECT sum(nb_evenements) FROM t WHERE annee <> 'Total') AS somme_des_annees
""")

,total,somme_des_annees
0,1016272,1016272.0


### Observations

- Le cube principal fait **2 987 190 lignes** (16 Mo) pour les 128 combinaisons de dimensions.
- La vérification marche : le total « toutes années » (1 016 272 entrées en défaillance) est bien la somme des années.
- ⚠️ L'année **2007** apparaît, avec seulement 1 810 entrées : ce sont des jugements de fin 2007 publiés début 2008. Elle est incomplète, tout comme 2026 (jusqu'à fin octobre).
- Le nombre d'**entreprises** est un peu plus petit que le nombre d'**événements** (par exemple 60 911 entreprises pour 65 220 entrées en 2025) : une même entreprise peut avoir plusieurs entrées dans l'année (ouverture puis extension, par exemple), et les annonces sans SIREN ne comptent pas comme entreprises.

## 5. Les cubes de détail (hiérarchies)

Le cube principal reste au niveau le plus large. Pour **descendre** dans une hiérarchie, on fait un petit cube par dimension avec `ROLLUP` :
- **temps** : année → trimestre → mois, croisé avec la nature exacte du jugement ;
- **territoire** : région → département → tribunal, par année ;
- **secteur** : section → division NAF, par année.

In [10]:
MESURES = "count(*) AS nb_evenements, count(DISTINCT siren) AS nb_entreprises"
cubes_detail = {
    "cube_temps": (["nature", "annee", "trimestre", "mois"], "CUBE (nature), ROLLUP (annee, trimestre, mois)"),
    "cube_territoire": (["annee", "region", "departement", "tribunal"], "CUBE (annee), ROLLUP (region, departement, tribunal)"),
    "cube_secteur": (["annee", "naf_section", "naf_division"], "CUBE (annee), ROLLUP (naf_section, naf_division)"),
}
for nom, (dims, groupes) in cubes_detail.items():
    con.execute(f"CREATE OR REPLACE TABLE {nom} AS SELECT categorie, {select_dims(dims)}, {MESURES} FROM faits GROUP BY categorie, {groupes}")
    print(nom, sql(f"SELECT count(*) AS n FROM {nom}")["n"].iloc[0], "lignes")

cube_temps 18990 lignes


cube_territoire 42059 lignes


cube_secteur 10062 lignes


In [11]:
# Exemple : les 10 tribunaux avec le plus d'entrées en défaillance en 2025
sql("""
SELECT region, departement, tribunal, nb_evenements FROM cube_territoire
WHERE categorie = 'entrée en défaillance' AND annee = '2025' AND tribunal <> 'Total'
ORDER BY nb_evenements DESC LIMIT 10
""")

,region,departement,tribunal,nb_evenements
0,Île-de-France,75,TRIBUNAL DE COMMERCE - PARIS,4820
1,Île-de-France,93,TRIBUNAL DE COMMERCE - BOBIGNY,2440
2,Auvergne-Rhône-Alpes,69,TRIBUNAL DE COMMERCE - LYON,2044
3,Nouvelle-Aquitaine,33,TRIBUNAL DE COMMERCE - BORDEAUX,1611
4,Île-de-France,94,TRIBUNAL DE COMMERCE - CRETEIL,1344
5,Provence-Alpes-Côte d'Azur,13,TRIBUNAL DE COMMERCE - MARSEILLE,1323
6,Île-de-France,91,TRIBUNAL DE COMMERCE - EVRY,1304
7,Île-de-France,92,TRIBUNAL DE COMMERCE - NANTERRE,1295
8,Occitanie,31,TRIBUNAL DE COMMERCE - TOULOUSE,1264
9,Île-de-France,77,TRIBUNAL DE COMMERCE - MEAUX,1222


### Observations

- Les cubes de détail sont petits : 18 990 lignes pour le temps, 42 059 pour le territoire, 10 062 pour le secteur.
- Grâce à l'harmonisation des noms de tribunaux (`nettoyage.ipynb`, décision BODACC 23), un tribunal qui a changé de nom n'est plus découpé en plusieurs morceaux. En 2025, le **Tribunal de Commerce de Paris** (devenu Tribunal des Activités Économiques) arrive en tête avec 4 820 entrées en défaillance, devant Bobigny (2 440) et Lyon (2 044).
- Un même tribunal peut apparaître sous un autre département pour une poignée d'annonces (une ou deux), quand le département de l'annonce ne correspond pas à celui du tribunal.

## 6. Les séries mensuelles pour les graphes

Les graphes des axes 8 et 9 ont besoin, pour chaque **division NAF** et chaque **département**, de la courbe mensuelle des entrées en défaillance. On les prépare ici au format « une ligne par mois, une colonne par division (ou département) ».

In [12]:
for nom, col in [("series_divisions_mois", "naf_division"), ("series_departements_mois", "departement")]:
    s = sql(f"""
        SELECT mois, {col}, count(*) AS entrees FROM faits
        WHERE categorie = 'entrée en défaillance' AND {col} <> 'non renseigné'
        GROUP BY ALL""")
    s = s.pivot_table(index="mois", columns=col, values="entrees", fill_value=0).sort_index()
    s.columns = s.columns.astype(str)
    s.to_parquet(DASH / f"{nom}.parquet")
    print(nom, s.shape)
s.iloc[-5:, :8]

series_divisions_mois (236, 84)
series_departements_mois (236, 103)


departement,01,02,03,04,05,06,07,08
mois,,,,,,,,
2026-06,31.0,27.0,23.0,12.0,11.0,95.0,26.0,11.0
2026-07,47.0,33.0,15.0,15.0,15.0,151.0,15.0,18.0
2026-08,18.0,4.0,10.0,6.0,0.0,6.0,3.0,0.0
2026-09,34.0,34.0,24.0,8.0,17.0,131.0,38.0,23.0
2026-10,43.0,23.0,22.0,15.0,7.0,85.0,12.0,11.0


### Observations

- On a les courbes mensuelles (236 mois, de fin 2007 à octobre 2026) pour **84 divisions NAF** et **103 départements**. Elles serviront aux graphes des axes 8 et 9.

## 7. Les entreprises actives (pour les taux)

Pour chaque année N de 2008 à 2026, on compte les entreprises **actives au 1er janvier N** : créées avant le 1er janvier N, et pas cessées à cette date. Pour une entreprise cessée, la date de cessation est sa `date_debut` (voir `sirene.ipynb`).

Pour aller vite, on regroupe d'abord SIRENE par année de création et année de cessation, puis on « déplie » sur les années. On peut le faire au niveau de l'année parce que tous les seuils sont des nombres entiers d'années : une entreprise créée en N-1 a forcément moins d'1 an au 1er janvier N, une créée en N-2 ou N-3 a entre 1 et 3 ans, etc.

In [13]:
con.execute(f"""
CREATE OR REPLACE TABLE sirene_groupes AS
SELECT year(date_creation) AS an_crea,
       CASE WHEN active THEN NULL ELSE year(date_debut) END AS an_cess,
       NOT active AND date_debut IS NULL AS cessation_inconnue,
       coalesce(d.region, 'non renseigné') AS region,
       coalesce(CAST(s.departement AS VARCHAR), 'non renseigné') AS departement,
       coalesce(CAST(naf_section AS VARCHAR), 'non renseigné') AS naf_section,
       coalesce(CAST(cj_famille AS VARCHAR), 'non renseigné') AS cj_famille,
       count(*) AS n
FROM read_parquet('{PROCESSED / "sirene_clean.parquet"}') s
LEFT JOIN dep_region d ON CAST(s.departement AS VARCHAR) = d.departement
GROUP BY ALL
""")
print(sql("SELECT count(*) AS groupes, sum(n) AS entreprises FROM sirene_groupes"))
sql("""SELECT sum(n) FILTER (WHERE an_crea IS NULL) AS sans_date_creation,
              sum(n) FILTER (WHERE cessation_inconnue) AS cessees_sans_date FROM sirene_groupes""")

   groupes  entreprises
0  2234058   30148401.0


,sans_date_creation,cessees_sans_date
0,1172957.0,404569.0


### Observations

- SIRENE se résume en 2,2 M de groupes (année de création × année de cessation × territoire × secteur × forme juridique), ce qui rend le calcul rapide.
- Ne sont **pas comptées** dans les actives : 1 172 957 entreprises sans date de création, et 404 569 entreprises cessées sans date de cessation.

In [14]:
DIMS_ACT = ["region", "naf_section", "cj_famille", "tranche_age"]
con.execute(f"""
CREATE OR REPLACE TABLE actives AS
WITH depli AS (
    SELECT a.annee, g.*, a.annee - g.an_crea AS d
    FROM sirene_groupes g
    JOIN (SELECT range AS annee FROM range(2008, 2027)) a
      ON g.an_crea < a.annee AND (g.an_cess IS NULL OR g.an_cess >= a.annee)
    WHERE NOT g.cessation_inconnue
)
SELECT CAST(annee AS VARCHAR) AS annee, {select_dims(DIMS_ACT)}, sum(n) AS nb_actives
FROM (SELECT *, CASE WHEN d <= 1 THEN '< 1 an' WHEN d <= 3 THEN '1-3 ans' WHEN d <= 5 THEN '3-5 ans'
                     WHEN d <= 10 THEN '5-10 ans' WHEN d <= 20 THEN '10-20 ans' ELSE '20 ans et +' END AS tranche_age
      FROM depli)
GROUP BY annee, CUBE ({", ".join(DIMS_ACT)})
""")
sql("""SELECT annee, nb_actives FROM actives
       WHERE region = 'Total' AND naf_section = 'Total' AND cj_famille = 'Total' AND tranche_age = 'Total'
       ORDER BY annee""")

,annee,nb_actives
0,2008,8450778.0
1,2009,8702640.0
2,2010,9198771.0
3,2011,9727277.0
4,2012,10166307.0
5,2013,10578056.0
6,2014,10973608.0
7,2015,11350596.0
8,2016,11706384.0
9,2017,11924690.0


### Observations

- Le nombre d'entreprises actives au 1er janvier **double presque** : de 8,5 M en 2008 à 16,2 M en 2026.

## 8. Le taux de défaillance

Taux = **entreprises entrées en défaillance** dans l'année (SIREN distincts) / **entreprises actives au 1er janvier**, en pour mille (‰). On le calcule pour toutes les combinaisons communes au cube principal et aux actives : année × région × section × famille juridique × tranche d'âge.

⚠️ Deux petites différences entre le numérateur et le dénominateur :
- le **numérateur** utilise le département de l'**annonce** (tribunal), le **dénominateur** celui du **siège** SIRENE. En général, c'est le même département ;
- l'âge du numérateur est pris **à la date du jugement**, celui du dénominateur **au 1er janvier**.

In [15]:
con.execute("""
CREATE OR REPLACE TABLE taux AS
SELECT c.annee, c.region, c.naf_section, c.cj_famille, c.tranche_age,
       c.nb_entreprises AS entreprises_defaillantes, a.nb_actives,
       1000.0 * c.nb_entreprises / a.nb_actives AS taux_pour_mille
FROM cube_principal c
JOIN actives a USING (annee, region, naf_section, cj_famille, tranche_age)
WHERE c.categorie = 'entrée en défaillance' AND c.tranche_taille = 'Total' AND c.type_personne = 'Total'
  AND c.annee <> 'Total'
""")
sql("""SELECT annee, entreprises_defaillantes, nb_actives, round(taux_pour_mille, 2) AS taux_pour_mille FROM taux
       WHERE region = 'Total' AND naf_section = 'Total' AND cj_famille = 'Total' AND tranche_age = 'Total'
       ORDER BY annee""")

,annee,entreprises_defaillantes,nb_actives,taux_pour_mille
0,2008,48041,8450778.0,5.68
1,2009,54857,8702640.0,6.30
2,2010,52364,9198771.0,5.69
3,2011,51925,9727277.0,5.34
4,2012,53118,10166307.0,5.22
5,2013,54746,10578056.0,5.18
6,2014,53084,10973608.0,4.84
7,2015,53461,11350596.0,4.71
8,2016,50246,11706384.0,4.29
9,2017,47621,11924690.0,3.99


In [16]:
# Exemple : taux par tranche d'âge en 2019
sql("""SELECT tranche_age, entreprises_defaillantes, nb_actives, round(taux_pour_mille, 2) AS taux_pour_mille FROM taux
       WHERE annee = '2019' AND region = 'Total' AND naf_section = 'Total' AND cj_famille = 'Total'
       ORDER BY taux_pour_mille DESC""")

,tranche_age,entreprises_defaillantes,nb_actives,taux_pour_mille
0,3-5 ans,9545,1066828.0,8.95
1,1-3 ans,10519,1300038.0,8.09
2,5-10 ans,11846,2490270.0,4.76
3,Total,45563,12785888.0,3.56
4,10-20 ans,8065,3146241.0,2.56
5,< 1 an,968,814991.0,1.19
6,20 ans et +,4626,3967520.0,1.17


### Observations

- Le taux de défaillance passe de **5,7 ‰ en 2008** (6,3 ‰ en 2009, le maximum) à **3,6 ‰ en 2019**, tombe à **1,8 ‰ en 2021**, puis remonte à **3,9 ‰ en 2024**.
- ⚠️ Comme le nombre d'entreprises actives a presque doublé, le **taux** baisse beaucoup plus que le **nombre** de défaillances. En 2024-2025, le nombre de défaillances dépasse celui de 2009, mais le taux reste plus bas.
- En 2019, le taux est le plus élevé pour les entreprises de **3-5 ans** (8,95 ‰) et **1-3 ans** (8,09 ‰). Il est très bas pour celles de **moins d'1 an** (1,19 ‰) et de **20 ans et plus** (1,17 ‰).

## 9. Enregistrement pour le dashboard

In [17]:
for t in ["cube_principal", "cube_temps", "cube_territoire", "cube_secteur", "actives", "taux"]:
    chemin = DASH / f"{t}.parquet"
    con.execute(f"COPY {t} TO '{chemin}' (FORMAT parquet)")
    print(f"{t:18s} {chemin.stat().st_size / 1e6:6.1f} Mo")
dep_region.to_parquet(DASH / "departements_regions.parquet", index=False)
con.close()
print(mem())

cube_principal       16.1 Mo
cube_temps            0.2 Mo
cube_territoire       0.3 Mo
cube_secteur          0.1 Mo
actives               2.0 Mo
taux                  1.9 Mo


process : 0.33 Go | disponible : 9.96 Go


### Observations

- Tous les fichiers du cube sont dans `data/dashboard/` et font **22 Mo au total**, dont 16 Mo pour le cube principal. Le dashboard pourra les charger sans problème de RAM.